# Lab 3C: Model Evaluation and Registration with Governance

Evaluate the base and fine-tuned Llama endpoints created in Lab 3B, record the
comparison in MLflow, and register the deployable fine-tuned artifact using the
**SageMaker Python SDK v3**.


## Set-Up

### Setup and Dependencies

Install required libraries and restart the kernel to ensure all packages are properly loaded.

In [ ]:
# Install the SageMaker SDK v3 package family and evaluation dependencies.
# Run once, wait for the kernel restart, then continue.
%pip install --upgrade --no-cache-dir -q     "sagemaker>=3,<4" "sagemaker-train>=1,<2" "sagemaker-serve>=1,<2"     "sagemaker-mlops>=1,<2" "datasets>=4.4,<6" "mlflow>=3,<4"     "sagemaker-mlflow>=0.5,<1" tiktoken "evaluate==0.4.0" rouge_score

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


In [ ]:
import importlib.metadata as metadata
import json
import time
from datetime import datetime
from packaging.version import Version

import boto3
import mlflow
import pandas as pd
from botocore.exceptions import ClientError

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.model_metrics import MetricsSource, ModelMetrics
from sagemaker.core.training.configs import Compute
from sagemaker.core.resources import Endpoint, ModelCard, ModelPackage, ModelPackageGroup
from sagemaker.core.shapes import ModelLifeCycle, ModelPackageModelCard
from sagemaker.serve.model_builder import ModelBuilder

sdk_version = Version(metadata.version("sagemaker"))
assert Version("3") <= sdk_version < Version("4"), (
    f"This notebook requires sagemaker>=3,<4; found {sdk_version}. "
    "Run the installation cell and wait for the kernel restart."
)

sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
sm_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"SageMaker SDK: {sdk_version}")
print(f"Execution role: {role}")
print(f"Default bucket: {bucket}")
print(f"Region:         {region}")


### Connect to MLflow App 

Retrieve the MLflow App detail that stores all experiment metadata from Lab 3. This connection allows us to access model artifacts, parameters, and metrics.

In [ ]:
try:
    response = sm_client.list_mlflow_apps(MaxResults=10)
    mlflow_apps = response.get('Summaries', [])
    
    if mlflow_apps:
        active_apps = [app for app in mlflow_apps if app['Status'] in ('Created', 'Updated')]
        
        if active_apps:
            mlflow_app_arn = active_apps[0]['Arn']
            mlflow_app_name = active_apps[0]['Name']
            print(f"✓ Found active MLflow App:")
            print(f"  Name: {mlflow_app_name}")
            print(f"  ARN: {mlflow_app_arn}")
        else:
            raise RuntimeError("No healthy MLflow App found (expected status Created or Updated)")
    else:
        raise RuntimeError("No MLflow Apps found in this region")
except Exception as e:
    raise RuntimeError(f"Could not resolve the workshop MLflow App: {e}") from e

### Load data from the MLFlow run

We use the `%store` magic command to retrieve variables saved from the previous notebook.

In [ ]:
# Restore the exact Lab 3B handoff. run_id avoids selecting another user's
# latest run from a shared MLflow experiment.
%store -r experiment_name
%store -r run_id
%store -r training_job_name
%store -r fine_tuned_model_endpoint_name
%store -r base_model_endpoint_name

print(f"Experiment:          {experiment_name}")
print(f"Run ID:              {run_id}")
print(f"Training job:        {training_job_name}")
print(f"Fine-tuned endpoint: {fine_tuned_model_endpoint_name}")
print(f"Base endpoint:       {base_model_endpoint_name}")

# SDK v3 typed endpoint resources used for invocation below.
base_model_endpoint = Endpoint.get(
    endpoint_name=base_model_endpoint_name,
    session=sess.boto_session,
    region=region,
)
fine_tuned_model_endpoint = Endpoint.get(
    endpoint_name=fine_tuned_model_endpoint_name,
    session=sess.boto_session,
    region=region,
)


In [ ]:
mlflow.set_tracking_uri(mlflow_app_arn)

# run_id is persisted by Lab 3B. Fall back to an exact training-job tag search
# only for runs produced before that handoff was added.
if not globals().get("run_id"):
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        filter_string=f"params.training_job_name = '{training_job_name}'",
        max_results=1,
    )
    if runs.empty:
        raise RuntimeError("Could not resolve the Lab 3B MLflow run")
    run_id = runs.iloc[0].run_id

run = mlflow.get_run(run_id)
print(f"Using MLflow run: {run_id}")


## Step 1: Model Evaluation

### Why Evaluate Models?

Before registering a model, we need to **quantify its performance** to:
- Validate that fine-tuning improved the model
- Establish baseline metrics for future comparisons
- Document performance for governance and compliance
- Make data-driven decisions about model deployment

### Evaluation Metrics for Summarization

We'll use standard NLP metrics:
- **BLEU**: Measures n-gram overlap between generated and reference text
- **ROUGE-1**: Unigram overlap (individual word matches)
- **ROUGE-2**: Bigram overlap (two-word phrase matches)
- **ROUGE-L**: Longest common subsequence (captures sentence structure)

Higher scores indicate better alignment with ground truth summaries.

### Load Evaluation Metrics

Import custom metric functions that calculate BLEU and ROUGE scores.

In [ ]:
from metrics import rouge1, rouge2, rougeL, bleu

### Retrieve Evaluation Dataset

We retrieve the evaluation dataset that was logged to MLflow during Lab 3. This demonstrates **data lineage** - we can trace exactly which data was used to evaluate this model.

In [ ]:
run = mlflow.get_run(run_id)
dataset_inputs = run.inputs.dataset_inputs

dataset_info = next(
    (d.dataset for d in dataset_inputs if any(tag.value == "evaluation" for tag in d.tags)),
    None
)
dataset_info

if dataset_info:
    source = mlflow.data.get_source(dataset_info)
    jsonl_path = source.load()
else:
    print("No dataset with context 'evaluation' found")

In [ ]:
import pandas as pd
evaluation_dataset = pd.read_json(jsonl_path, orient='records', lines=True)
evaluation_dataset.head()

### Define Endpoint Invocation Function

This helper function invokes SageMaker endpoints to get predictions from both the base model and fine-tuned model.

In [ ]:
def invoke_endpoint(endpoint, payload, *, accept_eula=False):
    """Invoke a typed SDK v3 Endpoint and decode its JSON response."""
    output = endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        accept="application/json",
        custom_attributes="accept_eula=true" if accept_eula else None,
    )
    body = output.body.read() if hasattr(output.body, "read") else output.body
    if isinstance(body, bytes):
        body = body.decode("utf-8")
    return json.loads(body) if isinstance(body, str) else body


def generated_text(response):
    """Normalize common text-generation response envelopes."""
    if isinstance(response, list) and len(response) == 1:
        response = response[0]
    if not isinstance(response, dict) or "generated_text" not in response:
        raise ValueError(f"Unexpected endpoint response shape: {response!r}")
    return response["generated_text"]


### Create Prompt Template

Define the instruction format used during fine-tuning. This ensures consistent prompt formatting when evaluating both models.

In [ ]:
import json

template = {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n",
    "completion": " {response}",
}

with open("template.json", "w") as f:
    json.dump(template, f)

### Generate Predictions from Both Models

This cell performs the actual evaluation by:
1. Sending each test example to both the base model and fine-tuned model
2. Collecting predictions from both models
3. Comparing them against ground truth responses

**Note**: This may take several minutes as we're invoking endpoints 20 times for each model (40 total invocations).

In [ ]:
from datasets import Dataset

test_dataset = Dataset.from_pandas(evaluation_dataset, preserve_index=False)
inputs, ground_truth_responses = [], []
responses_before_finetuning, responses_after_finetuning = [], []


def predict_and_collect(datapoint):
    input_output_demarkation_key = "\n\n### Response:\n"
    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], context=datapoint["context"]
        ) + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["response"])

    base_response = invoke_endpoint(base_model_endpoint, payload, accept_eula=True)
    responses_before_finetuning.append(generated_text(base_response))

    tuned_response = invoke_endpoint(fine_tuned_model_endpoint, payload)
    responses_after_finetuning.append(generated_text(tuned_response))


for datapoint in test_dataset.select(range(min(20, test_dataset.num_rows))):
    predict_and_collect(datapoint)

comparison_df = pd.DataFrame({
    "Inputs": inputs,
    "Ground Truth": ground_truth_responses,
    "Response from base model": responses_before_finetuning,
    "Response from fine-tuned model": responses_after_finetuning,
})
comparison_df.head()


### Prepare Data for Evaluation

Create DataFrames with predictions and ground truth for both models. MLflow's evaluate function requires this specific format.

In [ ]:
min_len = min(
    len(responses_before_finetuning),
    len(responses_after_finetuning),
    len(ground_truth_responses),
)
assert min_len > 0, "No endpoint responses were collected"


In [ ]:
df_before = pd.DataFrame({
    "predictions": responses_before_finetuning[:min_len],
    "targets": ground_truth_responses[:min_len]
})

df_after = pd.DataFrame({
    "predictions": responses_after_finetuning[:min_len],
    "targets": ground_truth_responses[:min_len]
})

### Calculate and Compare Metrics

Use MLflow's evaluate function to calculate metrics for both models. This:
- Automatically logs metrics to MLflow for tracking
- Creates separate runs for base and fine-tuned model evaluation
- Enables side-by-side comparison in the MLflow UI

**Expected Outcome**: The fine-tuned model should show higher scores across all metrics.

In [ ]:
from datetime import datetime
import logging

logging.getLogger("mlflow").setLevel(logging.ERROR)

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name=f"base-model-eval-{timestamp}"):
    result_before = mlflow.evaluate(
        data=df_before,
        targets="targets",
        predictions="predictions",
        extra_metrics=[bleu, rouge1, rouge2, rougeL]
    )

with mlflow.start_run(run_name=f"fine-tuned-model-eval-{timestamp}"):
    result_after = mlflow.evaluate(
        data=df_after,
        targets="targets",
        predictions="predictions",
        extra_metrics=[bleu, rouge1, rouge2, rougeL]

    )

print("\n=== Base Model ===")
print(f"BLEU:    {result_before.metrics['bleu']:.4f}")
print(f"ROUGE-1: {result_before.metrics['rouge1']:.4f}")
print(f"ROUGE-2: {result_before.metrics['rouge2']:.4f}")
print(f"ROUGE-L: {result_before.metrics['rougeL']:.4f}")

print("\n=== Fine-tuned Model ===")
print(f"BLEU:    {result_after.metrics['bleu']:.4f}")
print(f"ROUGE-1: {result_after.metrics['rouge1']:.4f}")
print(f"ROUGE-2: {result_after.metrics['rouge2']:.4f}")
print(f"ROUGE-L: {result_after.metrics['rougeL']:.4f}")

### Evaluation Results

✅ **The fine-tuned model outperforms the base model** across all metrics!

This quantitative evidence:
- Validates that fine-tuning was successful
- Provides metrics for governance documentation
- Justifies model registration and potential deployment
- Creates a baseline for future model versions

These metrics are now logged in MLflow and can be viewed in the tracking UI.

## Step 2: Model Registration

### Understanding SageMaker Model Registry

The SageMaker Model Registry is a **centralized repository** for managing ML models throughout their lifecycle. It provides:

**Key Features:**
- **Model Package Groups**: Logical grouping of related model versions
- **Versioning**: Automatic version tracking for each registered model
- **Approval Status**: Workflow states (Pending, Approved, Rejected)
- **Model Cards**: Embedded documentation with governance metadata
- **Lineage**: Links to training jobs, datasets, and experiments

**Governance Benefits:**
- **Audit Trail**: Complete history of model versions and approvals
- **Access Control**: IAM-based permissions for model deployment
- **Compliance**: Documentation required for regulatory requirements
- **Deployment Tracking**: Know which version is deployed where

### Model Registration Workflow
1. Retrieve model artifacts from MLflow
2. Register model with approval status
3. Create model card with business context
4. Set up lifecycle stages (Development → Staging → Production)

### Retrieve the Run details

### Get Container Image

For model registration, we need the **inference container image** used by JumpStart. This ensures the model can be deployed with the correct runtime environment.

In [ ]:
# Lab 3B records both training and inference metadata. Registration must use
# the inference image resolved by ModelBuilder, never the training image.
model_info = mlflow.artifacts.load_dict(run.info.artifact_uri + "/model_info.json")
container_image = model_info["inference_image_uri"]
inference_environment = model_info.get("inference_environment", {})
print(f"Inference image: {container_image}")
print(f"Environment keys: {sorted(inference_environment)}")


### Retrieve model artifact path

We need to extract the S3 location of the trained model from MLflow. This demonstrates **lineage tracking** - connecting MLflow experiments to Model Registry entries.

In [ ]:
model_path = model_info["model_artifact"]
assert model_path.startswith("s3://") and model_path.endswith(".tar.gz"), model_path
print(f"Model artifact location in Amazon S3: {model_path}")


### Create Model Package Group

A **Model Package Group** is a logical container for related model versions. Think of it as a "model family" where:
- All versions of the same model type are grouped together
- Each registration creates a new version (1, 2, 3, etc.)
- You can compare versions and track evolution over time

Example: `llama-summarization-models` might contain v1 (initial), v2 (improved), v3 (production).

In [ ]:
# Create a versioned Model Package Group using the SDK v3 resource API.
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
model_package_group_name = f"llama-summarization-models-{timestamp}"

try:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Fine-tuned Llama models for text summarization",
        session=sess.boto_session,
        region=region,
    )
    print(f"Created model package group: {model_package_group_name}")
except ClientError as error:
    if error.response["Error"]["Code"] not in ("ConflictException", "ResourceInUse"):
        raise
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name,
        session=sess.boto_session,
        region=region,
    )
    print(f"Using existing model package group: {model_package_group_name}")

model_package_group_arn = model_package_group.model_package_group_arn


**To view the Model Package Group you just created:**
1. Navigate to **SageMaker AI Studio**
2. In the left sidebar, scroll down and click **Models**
3. You'll see your model package group listed
   
![MLFlow Experiment](../../images/model-package-group.png)

### Register Model Package

Now we create the model package registration request with:
- **Model Package Group**: Logical grouping (e.g., "llama-summarization-models")
- **Container Image**: Inference runtime environment
- **Model Data URL**: S3 location of model artifacts
- **Model Card**: Governance metadata
- **Approval Status**: PendingManualApproval (requires explicit approval)

This creates a **versioned model entry** in the registry with complete lineage.

### Upload Metrics and Register Model

This cell performs the actual model registration:

**Step 1: Upload Metrics to S3**
- Package evaluation metrics in the required JSON format
- Upload to S3 so they can be referenced by the model package

**Step 2: Create Model Package**
- Links the model artifacts (model.tar.gz) from Lab 3
- Associates the container image for inference
- Attaches evaluation metrics
- Sets approval status to `PendingManualApproval`

**Approval Status Options:**
- `PendingManualApproval`: Requires explicit approval (recommended for production)
- `Approved`: Automatically approved (use for development/testing)
- `Rejected`: Explicitly rejected


In [ ]:
# Upload the fine-tuned evaluation report under a run-specific immutable key.
metrics_report = {
    "text_generation_metrics": {
        "bleu": {"value": result_after.metrics["bleu"], "standard_deviation": 0.0},
        "rouge1": {"value": result_after.metrics["rouge1"], "standard_deviation": 0.0},
        "rouge2": {"value": result_after.metrics["rouge2"], "standard_deviation": 0.0},
        "rougeL": {"value": result_after.metrics["rougeL"], "standard_deviation": 0.0},
    }
}
metrics_key = f"model-metrics/{run_id}/evaluation.json"
with open("evaluation.json", "w") as file:
    json.dump(metrics_report, file, indent=2)
s3_client.upload_file("evaluation.json", bucket, metrics_key)
metrics_s3_uri = f"s3://{bucket}/{metrics_key}"

# ModelBuilder.register is the SDK v3 registration API. The builder carries the
# exact inference image, model artifact, and environment recorded by Lab 3B.
registry_builder = ModelBuilder(
    image_uri=container_image,
    s3_model_data_url=model_path,
    env_vars=inference_environment,
    compute=Compute(instance_type="ml.g5.2xlarge", instance_count=1),
    role_arn=role,
    sagemaker_session=sess,
)
model_package_arn = registry_builder.register(
    model_package_group_name=model_package_group_name,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.g5.2xlarge"],
    transform_instances=["ml.g5.2xlarge"],
    model_metrics=ModelMetrics(
        model_statistics=MetricsSource(
            content_type="application/json",
            s3_uri=metrics_s3_uri,
        )
    ),
    approval_status="PendingManualApproval",
    description="Fine-tuned Llama 3.2 3B for text summarization",
    domain="NATURAL_LANGUAGE_PROCESSING",
    task="TEXT_GENERATION",
    customer_metadata_properties={
        "mlflow_run_id": run_id,
        "training_job_name": model_info["training_job_name"],
        "base_model_id": model_info["base_model_id"],
        "base_model_version": model_info["base_model_version"],
    },
)

# ModelBuilder.register returns the ARN for a versioned package. Rehydrate the
# typed SDK v3 resource for card, metadata, and lifecycle updates.
model_package = ModelPackage.get(
    model_package_name=model_package_arn,
    session=sess.boto_session,
    region=region,
)
# DescribeModelPackage omits ModelPackageName for versioned packages. The API
# accepts the ARN in that field, so preserve it for refresh/update operations.
model_package.model_package_name = model_package_arn

# Registration is asynchronous. Governance updates are accepted only after the
# package reaches Completed, so avoid racing the next cell.
for _ in range(60):
    if model_package.model_package_status in ("Completed", "Failed"):
        break
    time.sleep(5)
    model_package.refresh()
else:
    raise TimeoutError(f"Model Package did not complete within 5 minutes: {model_package_arn}")
if model_package.model_package_status == "Failed":
    raise RuntimeError(f"Model Package registration failed: {model_package_arn}")

print(f"Registered model package: {model_package_arn}")
print(f"Status:                   {model_package.model_package_status}")
print(f"Evaluation metrics:       {metrics_s3_uri}")


**To view it:**
1. Switch back to SageMaker AI Studio
2. In the Model Registry, click on the name of the model package group you created earlier.
3.  You will see the latest version: Version 1
4.  Click on it to view its details as per the following screenshot:
   
![MLFlow Experiment](../../images/model-package.png)

### Create Model Card

A **Model Card** is a structured document that provides transparency about a machine learning model. It's essential for:
- **Governance**: Documents model purpose, risks, and limitations
- **Compliance**: Meets regulatory requirements (e.g., EU AI Act, GDPR)
- **Transparency**: Helps stakeholders understand model behavior
- **Risk Management**: Identifies potential issues and mitigation strategies

### Model Card Sections

1. **Model Overview**: Creator, artifacts, version information
2. **Intended Uses**: Purpose, use cases, risk rating
3. **Business Details**: Problem statement, stakeholders, business unit
4. **Training Details**: Methodology, datasets, performance metrics
5. **Additional Information**: Ethical considerations, caveats, custom metadata

This model card will be embedded in the Model Registry entry, creating a **permanent governance record**.

### Create Model Card Content

Build the model card with comprehensive governance information. Each section serves a specific purpose:

**Model Overview**: Who created it, where artifacts are stored
**Intended Uses**: What problem it solves, risk assessment
**Business Details**: Business context and stakeholders
**Training Details**: How the model was trained
**Additional Information**: Ethical considerations, custom metadata

This information becomes part of the permanent model record.

In [ ]:
import mlflow
import json

model_card_content = {
    "model_overview": {
        "model_creator": "Data Science Team",
        "model_artifact": [model_path]  # You need to define s3_bucket and s3_key
    },
    "intended_uses": {
        "purpose_of_model": "Text summarisation",
        "intended_uses": "Answer to summarisation questions",
        "factors_affecting_model_efficiency": "Question complexity, technical domain coverage, input length",
        "risk_rating": "Low",
        "explanations_for_risk_rating": "Model provides informational summaries without making critical decisions"

    },
   "business_details": {
        "business_problem": "Improve efficiency of technical support and documentation access",
        "business_stakeholders": "Technical support team, Documentation team, End users",
        "line_of_business": "Customer Support & Knowledge Management"
    },
    "training_details": {
        "objective_function": {
            "function": "Instruction fine-tuning",
            "notes": "Fine-tuned Llama 3.2 3B on summarization tasks using Dolly dataset"
        },
        "training_observations": "Model trained to generate concise, accurate summaries of technical content"
    },
    "additional_information": {
        "ethical_considerations": "Ensure fair lending practices, avoid discriminatory outcomes",
        "caveats_and_recommendations": "Regular monitoring for model drift, periodic retraining with updated data",
        "custom_details": {
            "UseCaseId": "002",
            "UseCaseName": "Summarisation",
            "UseCaseStage": "Development"
        }
    }

}

# Save the model card
with open('model_card.json', 'w') as f:
    json.dump(model_card_content, f, indent=2)
print(model_card_content)
print("Model card has been created and saved.")


In [ ]:
print(model_package_arn)


In [ ]:
# Attach the inline model-package card with the typed SDK v3 update API.
model_package = model_package.update(
    model_card=ModelPackageModelCard(
        model_card_content=json.dumps(model_card_content),
        model_card_status="Draft",
    )
) or model_package
model_package.model_package_name = model_package_arn
print("✓ Inline model-package card attached")


In [ ]:
package_metadata = {
    "creator": "Data Science Team",
    "use_case": "Text Summarization",
    "business_problem": "Improve efficiency of technical support",
    "risk_rating": "Low",
    "model_type": "Fine-tuned Llama 3.2 3B",
    "mlflow_run_id": run_id,
}
model_package = model_package.update(
    customer_metadata_properties=package_metadata,
) or model_package
model_package.model_package_name = model_package_arn
print("✓ Governance metadata attached")


### Attach Model Card to the Model Package Group

Update the model card in SageMaker to associate it with the registered model package.

In [ ]:
# Create a standalone, versioned Model Card using the SDK v3 resource API.
model_card_name = f"model-card-{model_package_group_name}".replace("_", "-")
model_card = ModelCard.create(
    model_card_name=model_card_name,
    content=json.dumps(model_card_content),
    model_card_status="Draft",
    session=sess.boto_session,
    region=region,
)
print(f"Created model card: {model_card.model_card_arn}")


In [ ]:
model_card_arn = model_card.model_card_arn


In [ ]:
# Keep one complete metadata map so later updates cannot discard earlier keys.
package_metadata["model_card"] = model_card_arn
model_package = model_package.update(
    customer_metadata_properties=package_metadata,
) or model_package
model_package.model_package_name = model_package_arn
print("✓ Standalone Model Card ARN linked in package metadata")


## Step 3: Model Lifecycle Management

### Understanding Model Lifecycle Stages

SageMaker Model Registry supports **lifecycle stages** to track model progression through your ML workflow:

**Typical Stages:**
- **Development**: Model is being developed and tested
- **Staging**: Model is ready for pre-production testing
- **Production**: Model is approved for production deployment
- **Archived**: Model is retired from active use

**Stage Status:**
- **PendingApproval**: Awaiting review
- **Approved**: Cleared for use in this stage
- **Rejected**: Not approved for this stage

This creates a **formal approval workflow** ensuring only validated models reach production.

### Setting Lifecycle Stage

We'll mark this model as `Development/Approved` since it has passed evaluation but isn't ready for production yet.

### Update Model Lifecycle

This function updates the lifecycle stage of the registered model. You can modify the `Stage` and `StageStatus` values based on your organization's workflow.

**Customization Options:**
- Change `Stage` to: Development, Staging, Production, or Archived
- Change `StageStatus` to: PendingApproval, Approved, or Rejected
- Add `StageDescription` to document why the model is in this stage

In [ ]:
# Update the package lifecycle with the typed SDK v3 ModelLifeCycle shape.
model_package = model_package.update(
    model_life_cycle=ModelLifeCycle(
        stage="Development",
        stage_status="Approved",
        stage_description="Model trained and evaluated in the development environment",
    )
) or model_package
model_package.model_package_name = model_package_arn
print(f"✓ Lifecycle updated for {model_package_arn}: Development / Approved")


## Step 4: Delete Only the Lab 3B Real-Time Endpoints


Now that evaluation is complete, delete the two GPU endpoints created by Lab
3B. The cleanup is deliberately scoped to the stored endpoint names—it never
lists or deletes unrelated endpoints in the account. It also removes the endpoint
configurations and SageMaker Model resources after each endpoint is gone.


In [ ]:
def delete_endpoint_stack(endpoint_name):
    """Delete one known endpoint and its config/models; ignore if already absent."""
    try:
        endpoint_description = sm_client.describe_endpoint(EndpointName=endpoint_name)
    except ClientError as error:
        if error.response["Error"]["Code"] == "ValidationException":
            print(f"Endpoint already absent: {endpoint_name}")
            return
        raise

    config_name = endpoint_description["EndpointConfigName"]
    config = sm_client.describe_endpoint_config(EndpointConfigName=config_name)
    model_names = {variant["ModelName"] for variant in config.get("ProductionVariants", [])}

    endpoint = Endpoint.get(
        endpoint_name=endpoint_name,
        session=sess.boto_session,
        region=region,
    )
    print(f"Deleting endpoint: {endpoint_name}")
    endpoint.delete()
    sm_client.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)

    sm_client.delete_endpoint_config(EndpointConfigName=config_name)
    for model_name in model_names:
        sm_client.delete_model(ModelName=model_name)
    print(f"✓ Deleted endpoint, config, and model resources for {endpoint_name}")


In [ ]:
for endpoint_name in (base_model_endpoint_name, fine_tuned_model_endpoint_name):
    delete_endpoint_stack(endpoint_name)


## 🎉 Congratulations! Lab 3 Complete

### What You Accomplished

In this lab, you successfully:

1. ✅ **Evaluated model performance** using BLEU and ROUGE metrics
2. ✅ **Compared base vs. fine-tuned models** with quantitative evidence
3. ✅ **Created comprehensive model cards** with governance metadata
4. ✅ **Registered models** in SageMaker Model Registry with versioning
5. ✅ **Established approval workflows** with lifecycle management
6. ✅ **Linked evaluation metrics** to model registry entries

### Complete Governance Workflow

**Lab 3 + Lab 4 = End-to-End ML Governance**

Data Preparation → Fine-Tuning → MLflow Tracking → Model Evaluation → Model Registry → Approval Workflow

### Key Governance Capabilities

- **Lineage Tracking**: Trace models back to training data
- **Auditability**: Complete audit trail for compliance
- **Reproducibility**: Recreate any model version exactly
- **Compliance**: Model cards meet regulatory requirements
- **Version Control**: Compare and rollback models
- **Approval Workflows**: Prevent unauthorized deployments

### Viewing Your Work

- **SageMaker Console**: Navigate to Model Registry to view versions and model cards
- **MLflow UI**: Compare evaluation runs side-by-side
- **Programmatic Access**: Use boto3 to list model packages



### Next Steps

- Approve models for staging/production environments
- Implement CI/CD pipelines for automated deployment
- Set up model monitoring for drift detection
- Create governance dashboards

**Thank you for completing Lab 3!**